# Structured V_θ Sweep — Option 5 expressivity test

## Motivation

The PARFLM PR2 result (λ_V=10⁻⁴, 186 PPL) showed that with V_θ
regularisation, the learned potential is *small* (range 20) and the
attractor landscape is *low-rank* (K∗=4 basins).  This suggests that an
unrestricted MLP V_θ may be wasteful capacity, and a *structured*
parameterisation should suffice — with the major bonus that its
gradient ∇_h V_θ is computable analytically (one matvec, no
`autograd.grad` overhead).

This notebook tests four structured V_θ variants as drop-in
replacements for the MLP V_θ in regularised PARFLM, plus a reference
cell reproducing PR2.

## Cell structure

| Cell | V_θ kind | Attractors per context | Params @d=128 |
|------|----------|-----------------------|----------------|
| `SQ1` | Diagonal quadratic well | 1 | 33K |
| `SQ2` | Low-rank quadratic well (rank=8) | 1 | 165K |
| `SQ3` | Mixture of K=4 quadratic wells | **4** (matches PR2 K∗) | 133K |
| `SQ4` | Hybrid quadratic + small MLP residual | 1 + correction | 42K |
| `SQ5` | **Reference**: MLP V_θ (= PR2 reproduction) | extracted via GD | 66K |

All cells: PARFLM, L=8, λ_V=10⁻⁴, γ=0.10, TinyShakespeare, 4000 steps.

## Bonus interpretability metric

For SQ1–SQ4 the attractor centres are *explicit* in the model
parameters (μ(ξ) for the single-well variants, {μ_k(ξ)} for the
mixture).  We compare these analytical attractors against the
GD-extracted attractors from SQ5 to see whether the structured
parameterisation captures the same basin structure.

## 0. Environment setup + cell selector

In [ ]:
CELL = 'SQ1'      # one of: 'SQ1' | 'SQ2' | 'SQ3' | 'SQ4' | 'SQ5'
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula'
GDRIVE_OUT_REL  = 'semsimula_structured_vtheta'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.is_dir():
        shutil.rmtree(repo_data_dir)
    repo_data_dir.symlink_to(DATA_CACHE)
    print(f'data/ -> {DATA_CACHE}')

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf' / 'results' / 'structured_vtheta'
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)

SARF_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
sys.path.insert(0, str(SARF_DIR))
sys.path.insert(0, str(SARF_DIR / 'parf'))
sys.path.insert(0, str(SARF_DIR / 'sarf_mass_variant'))
sys.path.insert(0, str(SARF_DIR / 'energetic_minima'))

RESULTS_ROOT = GDRIVE_OUT
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run output dir = {RUN_DIR}')

## 1. GPU check

In [ ]:
import torch
import numpy as np

if torch.cuda.is_available():
    device = 'cuda'
    props = torch.cuda.get_device_properties(0)
    total_memory = props.total_memory / 1e9
    print(f'GPU: {props.name}  ({total_memory:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled (SPLM autograd.grad sensitivity)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('Using MPS')
else:
    device = 'cpu'
    print('WARNING: no GPU; training will be slow')

rng = np.random.default_rng(SEED)

## 2. Recipes (matched to PR2 PARFLM regularised baseline)

In [ ]:
STRUCTURED_RECIPES = {
    'SQ1': {
        'desc': 'Diagonal quadratic well (1 attractor)',
        'vtheta_kind': 'quadratic_diag',
        'vtheta_kwargs': {},
    },
    'SQ2': {
        'desc': 'Low-rank quadratic well (rank=8, 1 attractor)',
        'vtheta_kind': 'quadratic_lowrank',
        'vtheta_kwargs': {'rank': 8},
    },
    'SQ3': {
        'desc': 'Mixture of K=4 quadratic wells (matches PR2 K*=4)',
        'vtheta_kind': 'mixture',
        'vtheta_kwargs': {'K': 4, 'tau': 1.0},
    },
    'SQ4': {
        'desc': 'Hybrid: quadratic backbone + small MLP residual',
        'vtheta_kind': 'hybrid',
        'vtheta_kwargs': {'v_hidden': 32, 'v_depth': 2, 'alpha_init': 0.1},
    },
    'SQ5': {
        'desc': 'Reference: MLP V_theta (PR2 reproduction)',
        'vtheta_kind': 'mlp',
        'vtheta_kwargs': {},
    },
}
if CELL not in STRUCTURED_RECIPES:
    raise ValueError(f'CELL must be one of {list(STRUCTURED_RECIPES)}; got {CELL!r}')

recipe = STRUCTURED_RECIPES[CELL]

# Shared config (matches PR2 PARFLM)
VOCAB_SIZE = 50257
MAX_LEN    = 256
D          = 128
L          = 8
V_HIDDEN   = 128
V_DEPTH    = 3
DT         = 1.0
LAMBDA_V   = 1e-4
STEPS      = 4000
BATCH      = 16
BLOCK      = 128
INIT_GAMMA = 0.10
FIXED_GAMMA = None
TOP_K      = 16
SCORE_HEAD_HIDDEN = 32
V_PHI_KIND = 'structural'
V_PHI_D_TYPE = 16
V_PHI_D_ANGLE = 8
V_PHI_PHI_HIDDEN = 32
V_PHI_THETA_HIDDEN = 32
V_PHI_MLP_HIDDEN = 64
GUMBEL_TAU_INIT = 1.0
GUMBEL_TAU_MIN = 0.1
LR = 5e-4
WD = 0.01
WARMUP = 200
GRAD_CLIP = 1.0
EVAL_INTERVAL = 200
EVAL_ITERS = 40
LOG_INTERVAL = 50

print(f'Cell {CELL}: {recipe["desc"]}')
print(f'  vtheta_kind = {recipe["vtheta_kind"]}')
print(f'  vtheta_kwargs = {recipe["vtheta_kwargs"]}')
print(f'  d={D}  L={L}  lambda_V={LAMBDA_V}  steps={STEPS}')

## 3. Load TinyShakespeare

In [ ]:
from data_module import load_tiny_shakespeare, get_batch

train_ids, val_ids = load_tiny_shakespeare()
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

## 4. Build PARFLM with structured V_θ substitution

In [ ]:
from parf.model_parf_sparse import SparsePARFLM, SparsePARFConfig
from parf.model_structured_vtheta import (
    QuadraticWellVTheta, LowRankQuadraticVTheta,
    MixtureQuadraticVTheta, HybridQuadraticVTheta,
    validate_analytical_grad,
)
from sarf_mass_variant.model_sarf_mass import causal_cumulative_mean
import torch.nn.functional as F_torch

BUNDLED_LOGFREQ = SARF_DIR / 'sarf_mass_variant' / 'results' / 'logfreq_surprisal.npy'
DRIVE_LOGFREQ = RESULTS_ROOT / 'logfreq_surprisal_shakespeare.npy'

if BUNDLED_LOGFREQ.exists():
    LOGFREQ_PATH = BUNDLED_LOGFREQ
    print(f'Using bundled logfreq: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq: {LOGFREQ_PATH}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq; saved to {LOGFREQ_PATH}')

torch.manual_seed(SEED)

cfg = SparsePARFConfig(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
    v_phi_kind=V_PHI_KIND,
    v_phi_d_type=V_PHI_D_TYPE, v_phi_d_angle=V_PHI_D_ANGLE,
    v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
    v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
    v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_PATH),
    top_k=TOP_K,
    score_head_hidden=SCORE_HEAD_HIDDEN,
    gumbel_tau_init=GUMBEL_TAU_INIT,
    gumbel_tau_min=GUMBEL_TAU_MIN,
    init_gamma=INIT_GAMMA,
)
model = SparsePARFLM(cfg).to(device)

# Substitute V_theta if not the reference MLP
vtheta_kind = recipe['vtheta_kind']
vtheta_kwargs = recipe['vtheta_kwargs']
if vtheta_kind == 'quadratic_diag':
    model.V_theta = QuadraticWellVTheta(d=D, **vtheta_kwargs).to(device)
elif vtheta_kind == 'quadratic_lowrank':
    model.V_theta = LowRankQuadraticVTheta(d=D, **vtheta_kwargs).to(device)
elif vtheta_kind == 'mixture':
    model.V_theta = MixtureQuadraticVTheta(d=D, **vtheta_kwargs).to(device)
elif vtheta_kind == 'hybrid':
    model.V_theta = HybridQuadraticVTheta(d=D, **vtheta_kwargs).to(device)
elif vtheta_kind == 'mlp':
    pass  # Keep the default ScalarPotential MLP
else:
    raise ValueError(f'unknown vtheta_kind={vtheta_kind!r}')

# Quick sanity check on the substituted V_theta
if vtheta_kind != 'mlp':
    validate_analytical_grad(model.V_theta.cpu(), d=D, batch_shape=(2, 4))
    model.V_theta = model.V_theta.to(device)

n_total = sum(p.numel() for p in model.parameters())
n_vtheta = sum(p.numel() for p in model.V_theta.parameters())
n_vphi = sum(p.numel() for p in model.V_phi.parameters())
print(f'\nPARFLM with V_theta = {type(model.V_theta).__name__}')
print(f'  total params:  {n_total:,}')
print(f'  V_theta:       {n_vtheta:,}')
print(f'  V_phi:         {n_vphi:,}')

## 5. Training loop with V_θ regularisation

In [ ]:
import math, time, json

def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))

def tau_at(step):
    anneal_fraction = 0.8
    warm = int((1.0 - anneal_fraction) * STEPS)
    if step < warm:
        return GUMBEL_TAU_INIT
    if step >= STEPS:
        return GUMBEL_TAU_MIN
    progress = (step - warm) / max(STEPS - warm, 1)
    return GUMBEL_TAU_INIT + (GUMBEL_TAU_MIN - GUMBEL_TAU_INIT) * min(progress, 1.0)

def forward_with_vreg(model, x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = h_L @ model.E.weight.T
    loss_ntp = F_torch.cross_entropy(
        logits.reshape(-1, model.cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xi = causal_cumulative_mean(h_L.detach())
        V_vals = model.V_theta(xi, h_L)
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp
    return logits, loss, loss_ntp, v_reg_value

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

HAS_GUMBEL = hasattr(model, 'set_gumbel_tau')
opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()
log = []
best_ppl = float('inf')
t0 = time.time()

for step in range(STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)
    if HAS_GUMBEL:
        model.set_gumbel_tau(tau_at(step))
    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)
    _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    )
    opt.step()
    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        msg = (f'[{CELL}] step {step + 1:>5}/{STEPS}  '
               f'lr={lr_at(step):.2e}  '
               f'ntp={loss_ntp.item():.4f}  '
               f'v_reg={v_reg.item():.4f}  '
               f'total={loss.item():.4f}  '
               f'wall={time.time() - t0:.0f}s')
        print(msg)
    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        if val_ppl < best_ppl:
            best_ppl = val_ppl
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best_ppl={best_ppl:.2f}')
        log.append({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_ppl,
            'train_loss_ntp': loss_ntp.item(),
            'v_reg': v_reg.item(),
        })

print(f'\n[{CELL}] Training done.  wall={time.time() - t0:.0f}s  '
      f'final={log[-1]["val_ppl"]:.2f}  best={best_ppl:.2f}')

## 6. Save checkpoint + log

In [ ]:
full_tag = f'svth_{CELL}_{recipe["vtheta_kind"]}_seed{SEED}'
ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'recipe': recipe,
    'best_ppl': best_ppl,
    'step': STEPS,
    'vtheta_class': type(model.V_theta).__name__,
}, ckpt_path)
print(f'checkpoint: {ckpt_path}')

log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'
with open(log_path, 'w') as f:
    for e in log:
        f.write(json.dumps(e) + '\n')
print(f'log: {log_path}')

## 7. V_θ landscape diagnostics

In [ ]:
import matplotlib.pyplot as plt

v_samples = []
model.eval()
with torch.no_grad():
    for _ in range(10):
        xb, _ = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        with torch.enable_grad():
            out = model(x, return_trajectory=True)
        traj = out[2]
        h_L = traj[-1].to(device)
        xi = causal_cumulative_mean(h_L)
        V_vals = model.V_theta(xi, h_L).detach().cpu().numpy().ravel()
        v_samples.append(V_vals)

V_all = np.concatenate(v_samples)
landscape_stats = {
    'mean': float(V_all.mean()), 'std': float(V_all.std()),
    'min': float(V_all.min()), 'max': float(V_all.max()),
    'range': float(V_all.max() - V_all.min()),
}
print(f'V_theta on real trajectories ({CELL}):')
for k, v in landscape_stats.items():
    print(f'  {k:<6} = {v:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(V_all, bins=100, color='#3a6ea5', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('V_\u03b8(\u03be, h)')
ax.set_ylabel('count')
ax.set_title(f'{CELL} V_\u03b8 distribution ({recipe["desc"]})')
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_v_theta_hist.png', dpi=120)
plt.show()

with open(RUN_DIR / f'{full_tag}_landscape_stats.json', 'w') as f:
    json.dump(landscape_stats, f, indent=2)

## 8. Attractor extraction — analytical (SQ1–SQ4) vs GD (SQ5)

For the structured variants, the attractor centres are read directly
from `model.V_theta.attractor_centres(xi)`.  No 1500-step gradient
descent is needed.  For the reference MLP V_θ (SQ5), the existing GD
extraction protocol is used.

In [ ]:
from collections import Counter
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('gpt2')
except Exception as e:
    print(f'tokenizer unavailable ({e}); skipping centroid decoding')
    tok = None

PROMPTS = {
    'narrative':   'Once upon a time in a fair city',
    'mathematics': 'Let f be a continuous function on the interval',
    'scientific':  'The experiment was conducted in three phases',
    'dialogue':    'Yes, my lord, I will go at once',
    'code':        'def solve(arr, target):',
}

def decode_centroid(h_vec):
    if tok is None:
        return ['<no-tok>']
    with torch.no_grad():
        logits = h_vec @ model.E.weight.T
        probs = torch.softmax(logits, dim=-1)
        topk = torch.topk(probs, 5)
    return [
        (tok.decode([i.item()]).replace('\n', '\\n'), float(p.item()))
        for i, p in zip(topk.indices, topk.values)
    ]

model.eval()
attractor_report = {}
for name, prompt_text in PROMPTS.items():
    if tok is None:
        ids = torch.tensor([[1, 2, 3, 4, 5]], device=device)
    else:
        ids = torch.tensor(
            [tok.encode(prompt_text, add_special_tokens=False)],
            device=device,
        )
    with torch.no_grad():
        with torch.enable_grad():
            out = model(ids, return_trajectory=True)
        traj = out[2]
        h_last = traj[-1].to(device)
        xi = causal_cumulative_mean(h_last)[:, -1, :]
    
    print(f'\n=== {name}: "{prompt_text}" ===')
    if recipe['vtheta_kind'] != 'mlp' and hasattr(model.V_theta, 'attractor_centres'):
        centres = model.V_theta.attractor_centres(xi)
        centres = centres.squeeze(0)
        K = centres.shape[0]
        print(f'  Analytical attractor centres (K={K}):')
        decoded_per_basin = []
        for k in range(K):
            decoded = decode_centroid(centres[k])
            top_str = ', '.join(f'{repr(t)}:{p:.2f}' for t, p in decoded[:3])
            print(f'    A{k}: {top_str}')
            decoded_per_basin.append(decoded)
        attractor_report[name] = {
            'K': K, 'decoded': decoded_per_basin,
            'method': 'analytical',
        }
    else:
        print(f'  (SQ5 reference; use post-hoc GD extraction protocol)')
        attractor_report[name] = {'K': None, 'method': 'mlp_needs_gd'}

with open(RUN_DIR / f'{full_tag}_analytical_attractors.json', 'w') as f:
    json.dump(attractor_report, f, indent=2, default=str)
print(f'\nanalytical attractor report saved')

## 9. Cross-cell dashboard

In [ ]:
results = {}
for cell_name in ('SQ1', 'SQ2', 'SQ3', 'SQ4', 'SQ5'):
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    last = rows[-1] if rows else None
    best = min(r['val_ppl'] for r in rows) if rows else None
    ls_files = sorted(cell_dir.glob('*_landscape_stats.json'))
    ls = json.loads(ls_files[-1].read_text()) if ls_files else None
    results[cell_name] = {
        'desc': STRUCTURED_RECIPES[cell_name]['desc'],
        'val_ppl': last['val_ppl'] if last else None,
        'best_ppl': best,
        'landscape': ls,
    }

print(f'{"Cell":<6} {"Description":<48} {"best PPL":>10} {"final PPL":>10} '
      f'{"V range":>8}')
print('-' * 92)
print(f'{"":<6} {"PR2 reference (MLP V_theta, PARFLM \u03bb=1e-4)":<48} {"186.0":>10} '
      f'{"-":>10} {"20.0":>8}')
print(f'{"":<6} {"Attention baseline (TinyShakespeare)":<48} {"~150":>10} '
      f'{"-":>10} {"-":>8}')
print('-' * 92)

for cell_name, r in results.items():
    if r is None:
        desc = STRUCTURED_RECIPES[cell_name]['desc']
        print(f'{cell_name:<6} {desc:<48} {"\u2014":>10} {"\u2014":>10} '
              f'{"\u2014":>8}    (not run)')
        continue
    desc = r['desc']
    ppl_final = f"{r['val_ppl']:.1f}" if r['val_ppl'] else '\u2014'
    ppl_best = f"{r['best_ppl']:.1f}" if r['best_ppl'] else '\u2014'
    v_range = f"{r['landscape']['range']:.1f}" if r['landscape'] else '\u2014'
    print(f'{cell_name:<6} {desc:<48} {ppl_best:>10} {ppl_final:>10} '
          f'{v_range:>8}')

print(f'\nKey questions:')
print(f'  1. Does any structured variant match PR2 (186 PPL)?')
print(f'  2. Are analytical attractors interpretable as meaningful basins?')
print(f'  3. Is the mixture (SQ3) the best, matching the K*=4 PR2 observation?')